In [18]:
import timm
import torch
import numpy as np
import torch.nn as nn
import numpy as np
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
import time
model = timm.create_model('vit_base_patch16_224', pretrained=True)

config = resolve_data_config({}, model=model)
transform = create_transform(**config)

In [3]:
file_path = "pre_weights/dummy_input_4_196_768.bin"
with open(file_path, 'rb') as f:
    data = np.fromfile(f, dtype=np.float32).reshape(4, 196, 768)

In [4]:
torch.manual_seed(10)
inputs = torch.randn(4, 196, 768).reshape(4, 14, 14, 3, 16, 16).permute(0, 3, 1, 4, 2, 5).reshape(4, 3, 224, 224)

In [13]:

input = torch.tensor(data, dtype=torch.float16).reshape(4, 14, 14, 3, 16, 16).permute(0, 3, 1, 4, 2, 5).reshape(4, 3, 224, 224)
# inputs = torch.tensor(data, dtype=torch.float32).reshape(4, 14, 14, 3, 16, 16).permute(0, 3, 1, 4, 2, 5).reshape(4, 3, 224, 224)
type(inputs)

torch.dtype

In [14]:
for i in range(12):
    list(model.blocks.children())[i].attn.flash_attn = False

In [15]:
(dict(model.patch_embed.named_children())['proj'].weight).dtype

torch.float16

In [20]:
device = torch.device('cuda')
start = time.time()
inputs = torch.tensor(input.detach(), dtype=torch.float32)
# model = model.full()
modeld = model.to(device)


for i in range(100):
    
    input_d = inputs.to(device)
    output = modeld(input_d)
    output = output.to('cpu')
    print(f'output : {output}')
end = time.time()-start
print(f'time : {end}sec')


/tmp/ipykernel_22171/21709948.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  inputs = torch.tensor(input.detach(), dtype=torch.float32)


output : tensor([[-0.6506,  0.5699,  0.9909,  ..., -0.6971, -0.0040,  0.3011],
        [-0.5880,  0.3263,  0.6494,  ..., -0.4701,  0.1757,  0.2660],
        [-0.4764,  0.2062,  0.5736,  ..., -0.4387,  0.2734,  0.0986],
        [-0.6797,  0.2667,  0.9374,  ..., -0.5675, -0.0464,  0.2711]],
       grad_fn=<ToCopyBackward0>)
output : tensor([[-0.6506,  0.5699,  0.9909,  ..., -0.6971, -0.0040,  0.3011],
        [-0.5880,  0.3263,  0.6494,  ..., -0.4701,  0.1757,  0.2660],
        [-0.4764,  0.2062,  0.5736,  ..., -0.4387,  0.2734,  0.0986],
        [-0.6797,  0.2667,  0.9374,  ..., -0.5675, -0.0464,  0.2711]],
       grad_fn=<ToCopyBackward0>)
output : tensor([[-0.6506,  0.5699,  0.9909,  ..., -0.6971, -0.0040,  0.3011],
        [-0.5880,  0.3263,  0.6494,  ..., -0.4701,  0.1757,  0.2660],
        [-0.4764,  0.2062,  0.5736,  ..., -0.4387,  0.2734,  0.0986],
        [-0.6797,  0.2667,  0.9374,  ..., -0.5675, -0.0464,  0.2711]],
       grad_fn=<ToCopyBackward0>)
output : tensor([[-0.6506,  

In [9]:
output[:,18:25]

tensor([[3.5762, 1.8340, 1.5000, 5.7227, 4.8203, 4.7539, 1.8047],
        [3.0840, 1.6699, 1.2812, 5.1289, 3.9551, 4.0312, 1.9189],
        [3.5098, 1.8066, 1.5176, 5.2422, 4.4844, 4.3438, 2.1465],
        [3.3398, 1.6875, 1.4219, 5.4180, 4.6289, 4.4570, 1.9482]],
       dtype=torch.float16, grad_fn=<SliceBackward0>)

In [10]:
output[:,18:25]

tensor([[3.5762, 1.8340, 1.5000, 5.7227, 4.8203, 4.7539, 1.8047],
        [3.0840, 1.6699, 1.2812, 5.1289, 3.9551, 4.0312, 1.9189],
        [3.5098, 1.8066, 1.5176, 5.2422, 4.4844, 4.3438, 2.1465],
        [3.3398, 1.6875, 1.4219, 5.4180, 4.6289, 4.4570, 1.9482]],
       dtype=torch.float16, grad_fn=<SliceBackward0>)